## Combined Beauty research workflow

For catalog + reviews, evidence-backed 5 Whys and Good Question research cards, open [beauty_research_workflow.ipynb](beauty_research_workflow.ipynb). It runs the same EDA tools locally without an LLM/API call. The original demo below remains available.

# Local EDA Tools Agent example
Upstream notebook with corrected data paths and environment-based credentials.
Uses the demonstration churn dataset. LLM cells require API configuration and can incur charges.
The Beauty catalog analysis is a separate task.

# EDA Tools Agent

Exploratory Data Analysis is at the heart of data science. So we created the `EDA Tools Agent` to help you with your EDA. The `EDA Tools Agent` is a collection of tools that help you with your EDA:

- Missing Data with `missingno`
- EDA Reporting with `sweetviz`
- Correlation Analysis with `pytimetk` correlation funnel
- And more!

### Want To Become A Full-Stack Generative AI Data Scientist?

![Generative AI Data Scientist](../../img/become_a_generative_ai_data_scientist.jpg)

I teach Generative AI Data Science to help you build AI-powered data science apps. [**Register for my next Generative AI for Data Scientists workshop here.**](https://learn.business-science.io/ai-register)


# Table of Contents

1. [EDA Tools Agent](#eda-tools-agent)
2. [Load Libraries and Data](#load-libraries)
3. [Setup AI](#setup)
4. [Create The Agent](#create-the-agent)
5. [Usage](#usage)
    1. [What tools do you have access to?](#what-tools-do-you-have-access-to)
    2. [Get information on one of the agent's tools](#get-information-on-one-of-the-agents-tools)
6. [Tool Examples - How to use the agent's tools](#tool-examples)
    1. [Example 1: Describe data set tool](#example-1-describe-data-set-tool)
    2. [Example 2: Missing data tool](#example-2-missing-data-tool)
    3. [Example 3: Correlation Funnel Tool](#example-3-correlation-funnel-tool)
    4. [Example 4: Sweetviz Tool](#example-4-sweetviz-tool)
7. [Want To Become A Full-Stack Generative AI Data Scientist?](#want-to-become-a-full-stack-generative-ai-data-scientist)

### Load Libraries and Data <a id="load-libraries"></a>

We will load the libraries and data that we need for this notebook.

The dataset is the churn data from the `data` folder.

In [ ]:
# Common
from langchain_openai import ChatOpenAI
import pandas as pd
import os
import yaml

# Agent
from ai_data_science_team.ds_agents import EDAToolsAgent

# Helper functions
from ai_data_science_team.utils.matplotlib import matplotlib_from_base64
from ai_data_science_team.utils.plotly import plotly_from_dict
from ai_data_science_team.utils.html import open_html_file_in_browser

# Data
df = pd.read_csv("../vendor/ai-data-science-team/data/churn_data.csv")
df

### Setup AI <a id="setup"></a>

This section of code sets up the LLM inputs and the logging information. Logging is used to store AI-generated code and files during the AI Data Science Teams processing of files. 

*Important Note:* This example uses OpenAI's API. But any LLM can be used such as Anthropic or local LLMs with Ollama.

In [ ]:
# * Setup

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY in your shell before starting Jupyter to run the LLM example.")

llm = ChatOpenAI(model="gpt-4o-mini")
llm


### Create The Agent <a id="create-the-agent"></a>

Run this code to create an agent with `make_data_cleaning_agent()`.

In [ ]:
# Make a data loader agent
exploratory_agent = EDAToolsAgent(
    llm, 
    invoke_react_agent_kwargs={"recursion_limit": 10},
)
exploratory_agent.show(xray=1)

## Usage

Here are several examples of how to use the agent:


#### What tools do you have access to? Return a table.

In [ ]:
exploratory_agent.invoke_agent("What tools do you have access to? Return a table.")

exploratory_agent.get_ai_message(markdown=True)

#### Get information on one of the agent's tools.

In [ ]:
exploratory_agent.invoke_agent("Give me information on the correlation funnel tool.")

exploratory_agent.get_ai_message(markdown=True)

## Tool Examples - How to use the agent's tools

#### Example 1: Explain data tool

In [ ]:
exploratory_agent.invoke_agent(
    user_instructions="What are the first 5 rows of the data?",
    data_raw=df,
)
exploratory_agent.get_ai_message(markdown=True)

#### Example 2: Describe data set tool

In [ ]:
exploratory_agent.invoke_agent(
    user_instructions="Describe the dataset with the describe_dataset tool.",
    data_raw=df,
)

In [ ]:
# View the AI Message (for some reason markdown=True is not working well here so using HTML)
from IPython.display import display, HTML
msg = exploratory_agent.get_ai_message(markdown=False)
display(HTML(f"<pre>{msg}</pre>"))

In [ ]:
pd.DataFrame(exploratory_agent.get_artifacts()['describe_dataset']['describe_df'])

#### Example 3: Missing data tool


In [ ]:
exploratory_agent.invoke_agent(
    user_instructions="Visualize missing data in the dataset using the visualize_missing tool.",
    data_raw=df,
)

exploratory_agent.get_ai_message(markdown=True)

In [ ]:
# These artifacts are stored in the agent
exploratory_agent.get_artifacts()['visualize_missing'].keys()

In [ ]:
matplotlib_from_base64(
    exploratory_agent.get_artifacts()['visualize_missing']['matrix_plot']
)

In [ ]:
matplotlib_from_base64(
    exploratory_agent.get_artifacts()['visualize_missing']['bar_plot'],
)

In [ ]:
matplotlib_from_base64(
    exploratory_agent.get_artifacts()['visualize_missing']['heatmap_plot'],
)

#### Example 4: Correlation Funnel Tool

In [ ]:
exploratory_agent.invoke_agent(
    user_instructions="Use the correlation funnel tool to analyze the dataset. Use the Churn feature as the target. Use Churn=Yes as the bin index to compare against.",
    data_raw=df,
)
exploratory_agent.get_ai_message(markdown=True)

In [ ]:
exploratory_agent.get_artifacts()['generate_correlation_funnel'].keys()

In [ ]:
pd.DataFrame(exploratory_agent.get_artifacts()['generate_correlation_funnel']['correlation_data'])

In [ ]:
matplotlib_from_base64(exploratory_agent.get_artifacts()['generate_correlation_funnel']['plot_image'])

In [ ]:
plotly_from_dict(exploratory_agent.get_artifacts()['generate_correlation_funnel']['plotly_figure'])

![Correlation Funnel](../img/ds_agents/correlation_funnel.jpg)

#### Example 5: Sweetviz Tool

In [ ]:
exploratory_agent.invoke_agent(
    user_instructions="Generate a Sweetviz report for the dataset. Use the Churn feature as the target.",
    data_raw=df,
)
exploratory_agent.get_ai_message(markdown=True)

In [ ]:
exploratory_agent.get_artifacts()['generate_sweetviz_report'].keys()

In [ ]:
open_html_file_in_browser(exploratory_agent.get_artifacts()['generate_sweetviz_report']['report_file'])

![Sweetviz Report](../img/ds_agents/sweetviz_report.jpg)

### Want To Become A Full-Stack Generative AI Data Scientist?

![Generative AI Data Scientist](../../img/become_a_generative_ai_data_scientist.jpg)

I teach Generative AI Data Science to help you build AI-powered data science apps. [**Register for my next Generative AI for Data Scientists workshop here.**](https://learn.business-science.io/ai-register)